# Semantic Drift and Return Predictability
## Evidence from S&P 500 10-K Filings (2005–2024)

**Signal**: Year-over-year cosine distance of `all-MiniLM-L6-v2` embeddings of Item 1 (Business Description).  
Higher semantic drift = greater strategic repositioning in annual language.

| Section | Content |
|---|---|
| 5 | Descriptive statistics |
| 6 | Quintile portfolio sorts (Table 2) |
| 7 | Fama-MacBeth regressions with Newey-West SEs (Table 3) |
| 8 | Calendar-time long/short portfolio — Carhart 4-factor alpha (Table 4) |
| 9 | Performance statistics and figures |

**Citation**: Cohen, Malloy & Nguyen (2020, JF) forward-return methodology;  
Fama & MacBeth (1973, JPE) cross-sectional regression framework.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 1. Configuration

In [ ]:
import os

CONFIG = {
    'drift_path':          '/content/drive/MyDrive/FML_project_4/semantic_drift_minilm.parquet',
    'returns_path':        '/content/drive/MyDrive/FML_project_4/lazy_prices_dataset.csv',
    'usable_pairs_path':   '/content/drive/MyDrive/FML_project_4/usable_pairs.parquet',
    'output_folder':       '/content/drive/MyDrive/FML_project_4',
    # Newey-West lags for Fama-MacBeth (6 months captures quarterly earnings persistence)
    'nw_lags':    6,
    # Symmetric winsorization at 1st/99th percentile
    'winsor':     0.01,
    # Minimum cross-sectional observations per month for FM regression
    'min_xsec':   30,
    'start_year': 2005,
    'end_year':   2024,
}
print('Config OK')

## 2. Dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyarrow', 'pandas-datareader', 'statsmodels', 'tqdm'], check=True)

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from scipy import stats
from scipy.stats.mstats import winsorize as _winsorize
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({
    'figure.dpi': 150, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('Dependencies OK')

## 3. Load Data

In [ ]:
# ── Semantic drift signal ─────────────────────────────────────────────
drift_raw = pd.read_parquet(CONFIG['drift_path'])
drift_raw['cik']  = drift_raw['cik'].astype(str).str.strip().str.lstrip('0')
drift_raw['year'] = drift_raw['year'].astype('Int64')
# Keep only consecutive year pairs (gap > 1 means a missing filing in between)
drift_raw = drift_raw[
    drift_raw['year'].astype(int) - drift_raw['year_prev'].astype(int) == 1
].dropna(subset=['semantic_drift'])
drift_raw = drift_raw[drift_raw['year'].between(CONFIG['start_year'], CONFIG['end_year'])]
print(f'Drift: {len(drift_raw):,} obs  |  {drift_raw["cik"].nunique():,} CIKs  |  '
      f'years {drift_raw["year"].min()}\u2013{drift_raw["year"].max()}')

# ── Returns panel ─────────────────────────────────────────────────────
returns = pd.read_csv(CONFIG['returns_path'])
returns['fdate']          = pd.to_datetime(returns['fdate'], errors='coerce')
returns['return_month']   = pd.to_datetime(returns['return_month'], errors='coerce')
returns['cik']            = returns['cik'].astype(str).str.strip().str.lstrip('0')
returns['year']           = returns['fdate'].dt.year.astype('Int64')
returns['monthly_return'] = pd.to_numeric(returns['monthly_return'], errors='coerce')
for col in ['at', 'sale', 'ni']:
    if col in returns.columns:
        returns[col] = pd.to_numeric(returns[col], errors='coerce')
print(f'Returns: {len(returns):,} rows  |  {returns["cik"].nunique():,} CIKs')
print(f'Columns: {list(returns.columns)}')

# ── Usable pairs (S&P 500 membership filter) ─────────────────────────
usable = pd.read_parquet(CONFIG['usable_pairs_path'])
usable['cik']  = usable['cik'].astype(str).str.strip().str.lstrip('0')
usable['year'] = usable['year'].astype('Int64')
usable_set     = set(zip(usable['cik'], usable['year'].astype(int)))
print(f'Usable (cik, year) pairs: {len(usable_set):,}')

## 4. Panel Construction

**Forward-return alignment** (Cohen, Malloy & Nguyen 2020):  
For each filing with date `fdate`, the return window is `(fdate, fdate + 12 months]`.  
This ensures no return information predates the public release of the signal.

**Controls** computed from the full monthly return panel (backward-looking):  
- *Momentum*: cumulative return months t\u221212 to t\u22122 (Jegadeesh & Titman 1993)  
- *Reversal*: return in month t\u22121 (short-term reversal)  
- *Size*: log(total assets) at most recent fiscal year-end (proxy for log market cap)  
- *Profitability*: net income / total assets (ROA)

In [ ]:
# ── Step 1: one filing record per (cik, year) ─────────────────────────
filing_meta = (
    returns.sort_values('fdate')
           .drop_duplicates(subset=['cik', 'year'], keep='first')
    [['cik', 'year', 'fdate', 'at', 'ni']].copy()
)
filing_meta = filing_meta[
    filing_meta.apply(lambda r: (r['cik'], int(r['year'])) in usable_set, axis=1)
]

# ── Step 2: attach drift signal ───────────────────────────────────────
signal_meta = filing_meta.merge(
    drift_raw[['cik', 'year', 'semantic_drift']],
    on=['cik', 'year'], how='inner'
).dropna(subset=['semantic_drift'])
signal_meta['fdate_period'] = signal_meta['fdate'].dt.to_period('M')
signal_meta['size'] = np.where(
    signal_meta['at'].notna() & (signal_meta['at'] > 0),
    np.log(signal_meta['at']), np.nan
)
signal_meta['profitability'] = np.where(
    signal_meta['ni'].notna() & signal_meta['at'].notna() & (signal_meta['at'] > 0),
    signal_meta['ni'] / signal_meta['at'], np.nan
)
print(f'Signal filings: {len(signal_meta):,}  ({signal_meta["cik"].nunique():,} CIKs)')

# ── Step 3: backward-looking controls from full monthly panel ─────────
monthly = (
    returns[['cik', 'return_month', 'monthly_return']]
    .drop_duplicates(subset=['cik', 'return_month'])
    .sort_values(['cik', 'return_month'])
    .reset_index(drop=True)
)
monthly['monthly_return'] = pd.to_numeric(monthly['monthly_return'], errors='coerce')
monthly['log_r']  = np.log1p(monthly['monthly_return'].fillna(0))
monthly['ret_period'] = monthly['return_month'].dt.to_period('M')
g = monthly.groupby('cik')
monthly['reversal'] = g['monthly_return'].shift(1)
monthly['momentum'] = np.expm1(
    g['log_r'].transform(lambda x: x.shift(2).rolling(11, min_periods=8).sum())
)
print(f'Monthly panel: {len(monthly):,} rows')

# ── Step 4: vectorised forward-return panel ───────────────────────────
panel = signal_meta[['cik', 'year', 'fdate', 'fdate_period',
                      'semantic_drift', 'size', 'profitability']].merge(
    monthly[['cik', 'return_month', 'ret_period', 'monthly_return', 'reversal', 'momentum']],
    on='cik', how='inner'
)
panel = panel[
    (panel['ret_period'] > panel['fdate_period']) &
    (panel['ret_period'] <= panel['fdate_period'] + 12)
].drop(columns=['fdate_period', 'ret_period']).copy()

print(f'Panel: {len(panel):,} firm-month obs  |  '
      f'{panel["cik"].nunique():,} CIKs  |  '
      f'{panel["return_month"].min().date()} \u2013 {panel["return_month"].max().date()}')

## 5. Winsorise & Standardise

In [ ]:
def winsorize(s: pd.Series, p: float) -> pd.Series:
    out = s.copy()
    lo, hi = s.quantile(p), s.quantile(1 - p)
    out = out.clip(lower=lo, upper=hi)
    return out

WIN_COLS = ['semantic_drift', 'monthly_return', 'momentum', 'reversal', 'size', 'profitability']
for col in WIN_COLS:
    if col in panel.columns:
        panel[col] = winsorize(panel[col], CONFIG['winsor'])

# Cross-sectionally standardise signal each month for FM regression
panel['drift_std'] = (
    panel.groupby('return_month')['semantic_drift']
         .transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
)

# Quintile rank within each filing year for portfolio sorts
panel['quintile'] = (
    panel.groupby('year')['semantic_drift']
         .transform(lambda x: pd.qcut(x.rank(method='first'), 5,
                                       labels=[1, 2, 3, 4, 5]))
         .astype(int)
)

CONTROLS = [c for c in ['momentum', 'reversal', 'size', 'profitability']
            if c in panel.columns and panel[c].notna().mean() > 0.3]
print(f'Controls available: {CONTROLS}')
print(panel[WIN_COLS].describe().round(4))

## 6. Descriptive Statistics

In [ ]:
desc_vars = [
    ('semantic_drift',  'Semantic Drift'),
    ('monthly_return',  'Monthly Return'),
    ('momentum',        'Momentum (12\u22122m)'),
    ('reversal',        'Reversal (1m)'),
    ('size',            'log(Assets)'),
    ('profitability',   'Profitability (ROA)'),
]

rows = []
for col, label in desc_vars:
    if col not in panel.columns:
        continue
    s = panel[col].dropna()
    rows.append({
        'Variable':   label,
        'N':          f'{len(s):,}',
        'Mean':       f'{s.mean():.4f}',
        'Std':        f'{s.std():.4f}',
        'p10':        f'{s.quantile(.10):.4f}',
        'Median':     f'{s.quantile(.50):.4f}',
        'p90':        f'{s.quantile(.90):.4f}',
        'Skew':       f'{s.skew():.3f}',
    })

tbl1 = pd.DataFrame(rows)
print('\nTable 1 \u2014 Summary Statistics')
print('=' * 80)
print(tbl1.to_string(index=False))
print('Note: All continuous variables winsorised at 1st/99th percentile.')

# ── Figure 1: time series of mean semantic drift ──────────────────────
annual = (
    drift_raw[drift_raw['year'].between(CONFIG['start_year'], CONFIG['end_year'])]
    .groupby('year')['semantic_drift']
    .agg(mean='mean', p25=lambda x: x.quantile(.25), p75=lambda x: x.quantile(.75),
         count='count')
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.fill_between(annual['year'], annual['p25'], annual['p75'], alpha=0.2, color='#1f77b4', label='IQR')
ax.plot(annual['year'], annual['mean'], marker='o', color='#1f77b4', linewidth=1.8, label='Mean')
ax.set_title('Semantic Drift Over Time (Item 1, MiniLM-L6-v2)', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('1 \u2212 cosine similarity')
ax.legend(fontsize=9); ax.grid(linestyle='--', alpha=0.4)

ax = axes[1]
ax.hist(drift_raw['semantic_drift'], bins=50, color='#2ca02c', edgecolor='white', alpha=0.85)
ax.axvline(drift_raw['semantic_drift'].median(), color='red', linestyle='--',
           label=f'Median = {drift_raw["semantic_drift"].median():.3f}')
ax.set_title('Pooled Distribution of Semantic Drift', fontsize=11)
ax.set_xlabel('Semantic Drift'); ax.set_ylabel('Frequency')
ax.legend(fontsize=9); ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Figure 1 \u2014 Semantic Drift Signal Diagnostics', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'fig1_drift_diagnostics.pdf'),
            bbox_inches='tight')
plt.show()

## 7. Quintile Portfolio Sorts

Firms are sorted into quintiles based on the semantic drift signal measured at the
10-K filing year. Equal-weighted buy-and-hold returns are computed over the
12-month forward window beginning the month after the filing date.

In [ ]:
# Monthly L/S time series for proper t-stat
monthly_q = (
    panel.groupby(['return_month', 'quintile'])['monthly_return']
         .mean().unstack('quintile')
)

def tstat_series(s: pd.Series) -> float:
    s = s.dropna()
    return s.mean() / (s.std() / np.sqrt(len(s)))

print('\nTable 2 \u2014 Quintile Portfolio Sorts on Semantic Drift')
print('Dependent variable: monthly return (%) over 12-month post-filing window')
print('=' * 72)
print(f'{"Quintile":<12} {"Mean Ret (%)":>13} {"Ann. Ret (%)":>13} {"Ann. Vol (%)":>13} {"t-stat":>10}')
print('-' * 72)

q_stats = {}
for q in [1, 2, 3, 4, 5]:
    if q not in monthly_q.columns:
        continue
    ts  = monthly_q[q].dropna()
    mn  = ts.mean()
    ann = (1 + mn)**12 - 1
    vol = ts.std() * np.sqrt(12)
    t   = tstat_series(ts)
    q_stats[q] = {'mean': mn, 'ann': ann, 'vol': vol, 't': t, 'ts': ts}
    st  = '***' if abs(t) > 2.576 else ('**' if abs(t) > 1.960 else ('*' if abs(t) > 1.645 else ''))
    lbl = f'Q{q} (Low drift)' if q == 1 else (f'Q{q} (High drift)' if q == 5 else f'Q{q}')
    print(f'{lbl:<12} {mn*100:>12.3f}%  {ann*100:>12.3f}%  {vol*100:>12.3f}%  ({t:>6.2f}){st}')

print('-' * 72)
if 1 in q_stats and 5 in q_stats:
    ls_ts   = (monthly_q[5] - monthly_q[1]).dropna()
    ls_mean = ls_ts.mean()
    ls_ann  = (1 + ls_mean)**12 - 1
    ls_vol  = ls_ts.std() * np.sqrt(12)
    ls_sr   = ls_ts.mean() / ls_ts.std() * np.sqrt(12)
    ls_t    = tstat_series(ls_ts)
    ls_st   = '***' if abs(ls_t) > 2.576 else ('**' if abs(ls_t) > 1.960 else ('*' if abs(ls_t) > 1.645 else ''))
    print(f'{"Q5 \u2212 Q1":<12} {ls_mean*100:>12.3f}%  {ls_ann*100:>12.3f}%  {ls_vol*100:>12.3f}%  ({ls_t:>6.2f}){ls_st}')
    print('=' * 72)
    print(f'Long/Short annualised Sharpe: {ls_sr:.3f}')
print('t-statistics from monthly time series. *** p<0.01, ** p<0.05, * p<0.10.')

In [ ]:
# ── Figure 2: quintile mean returns bar chart ─────────────────────────
if q_stats:
    qs  = sorted(q_stats.keys())
    ann_rets = [q_stats[q]['ann'] * 100 for q in qs]
    colors   = ['#d62728', '#ff7f0e', '#7f7f7f', '#2ca02c', '#1f77b4']

    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.bar(qs, ann_rets, color=colors[:len(qs)], alpha=0.85, edgecolor='white', width=0.6)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    for bar, v in zip(bars, ann_rets):
        ax.text(bar.get_x() + bar.get_width() / 2, v + np.sign(v) * 0.3,
                f'{v:.1f}%', ha='center', va='bottom' if v >= 0 else 'top', fontsize=9)
    ax.set_xticks(qs)
    ax.set_xticklabels([f'Q{q}' for q in qs])
    ax.set_xlabel('Semantic Drift Quintile (Q1 = lowest, Q5 = highest)')
    ax.set_ylabel('Equal-weighted Annual Return (%)')
    ax.set_title('Figure 2 \u2014 Quintile Portfolio Returns on Semantic Drift', fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_folder'], 'fig2_quintile_sort.pdf'),
                bbox_inches='tight')
    plt.show()

## 8. Fama-MacBeth Cross-Sectional Regressions

Each month, a cross-sectional OLS regression is run:

$$R_{i,t+1} = \alpha_t + \beta_t \cdot \text{SemanticDrift}_{i,\tau(i,t)} + \gamma_t \mathbf{X}_{i,t} + \varepsilon_{i,t+1}$$

where $\tau(i,t)$ indexes the most recent 10-K filing before month $t$.  
Time-series means of $\hat{\beta}_t$ are reported with Newey-West (1987) corrected standard errors
using $L = 6$ lags to account for serial correlation induced by the 12-month holding period.

In [ ]:
def nw_variance(series: np.ndarray, n_lags: int) -> float:
    """Newey-West long-run variance of a 1-D time series."""
    T = len(series)
    d = series - series.mean()
    v = float(np.dot(d, d)) / T
    for lag in range(1, n_lags + 1):
        w   = 1.0 - lag / (n_lags + 1)          # Bartlett kernel
        cov = float(np.dot(d[lag:], d[:-lag])) / T
        v  += 2.0 * w * cov
    return max(v, 1e-20)


def fama_macbeth(
    df:           pd.DataFrame,
    dep:          str,
    signal:       str,
    controls:     list,
    date_col:     str,
    nw_lags:      int,
    min_obs:      int,
) -> pd.DataFrame:
    """
    Fama-MacBeth (1973) cross-sectional OLS.
    Returns DataFrame: Variable, Coef, t_stat, p_val, N_months, Avg_N, Avg_R2.
    """
    indep     = [signal] + controls
    all_cols  = [dep] + indep
    coefs, r2s, ns = [], [], []

    for month, sub in df.dropna(subset=all_cols).groupby(date_col):
        if len(sub) < min_obs:
            continue
        y = sub[dep].values
        X = np.column_stack([np.ones(len(sub))] + [sub[v].values for v in indep])
        try:
            coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        except np.linalg.LinAlgError:
            continue
        yhat = X @ coef
        ss_tot = ((y - y.mean())**2).sum()
        r2s.append(1 - ((y - yhat)**2).sum() / ss_tot if ss_tot > 0 else np.nan)
        coefs.append(coef)
        ns.append(len(sub))

    if not coefs:
        raise ValueError('No cross-sections passed min_obs filter.')

    arr  = np.array(coefs)         # (T, K)
    T    = arr.shape[0]
    mean = arr.mean(axis=0)        # (K,)
    tst  = np.array([
        mean[k] / np.sqrt(nw_variance(arr[:, k], nw_lags) / T)
        for k in range(arr.shape[1])
    ])
    pval = 2 * stats.t.sf(np.abs(tst), df=T - 1)

    return pd.DataFrame({
        'Variable':  ['const'] + indep,
        'Coef':       mean,
        't_stat':     tst,
        'p_val':      pval,
        'N_months':   T,
        'Avg_N':      int(np.mean(ns)),
        'Avg_R2':     np.nanmean(r2s),
    })

print('Fama-MacBeth function defined.')

In [ ]:
nw  = CONFIG['nw_lags']
mxs = CONFIG['min_xsec']

# Model 1: raw signal, no controls
fm1 = fama_macbeth(panel, 'monthly_return', 'semantic_drift', [],
                   'return_month', nw, mxs)

# Model 2: standardised signal, no controls
fm2 = fama_macbeth(panel, 'monthly_return', 'drift_std', [],
                   'return_month', nw, mxs)

# Model 3: standardised signal + all available controls
fm3 = fama_macbeth(panel.dropna(subset=CONTROLS), 'monthly_return',
                   'drift_std', CONTROLS, 'return_month', nw, mxs)

def stars(p: float) -> str:
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))

def fmt(row: pd.Series, scale: float = 100.0) -> tuple:
    """Return (coefficient string with stars, t-stat string)."""
    return (f'{row["Coef"]*scale:.4f}{stars(row["p_val"])}',
            f'({row["t_stat"]:.2f})')

VAR_LABELS = {
    'semantic_drift': 'Semantic Drift',
    'drift_std':      'Semantic Drift (std)',
    'momentum':       'Momentum (12\u22122m)',
    'reversal':       'Reversal (1m)',
    'size':           'log(Assets)',
    'profitability':  'Profitability (ROA)',
    'const':          'Intercept',
}

all_vars = ['semantic_drift', 'drift_std'] + CONTROLS + ['const']

print('\nTable 3 \u2014 Fama-MacBeth Cross-Sectional Regressions')
print('Dependent variable: monthly stock return (%)')
print(f'Newey-West standard errors ({nw} lags)')
print('=' * 76)
print(f'{"Variable":<24} {"(1) Raw":>16} {"(2) Std":>16} {"(3) + Controls":>16}')
print('-' * 76)

for var in all_vars:
    label = VAR_LABELS.get(var, var)
    r1 = fm1[fm1['Variable'] == var]
    r2 = fm2[fm2['Variable'] == var]
    r3 = fm3[fm3['Variable'] == var]
    c1 = fmt(r1.iloc[0]) if len(r1) else ('', '')
    c2 = fmt(r2.iloc[0]) if len(r2) else ('', '')
    c3 = fmt(r3.iloc[0]) if len(r3) else ('', '')
    if not any([c1[0], c2[0], c3[0]]):
        continue
    print(f'{label:<24} {c1[0]:>16} {c2[0]:>16} {c3[0]:>16}')
    print(f'{"":<24} {c1[1]:>16} {c2[1]:>16} {c3[1]:>16}')

print('-' * 76)
print(f'{"Months (T)":<24} {int(fm1["N_months"].iloc[0]):>16} '
      f'{int(fm2["N_months"].iloc[0]):>16} {int(fm3["N_months"].iloc[0]):>16}')
print(f'{"Avg N per month":<24} {int(fm1["Avg_N"].iloc[0]):>16} '
      f'{int(fm2["Avg_N"].iloc[0]):>16} {int(fm3["Avg_N"].iloc[0]):>16}')
print(f'{"Avg R\u00b2":<24} {fm1["Avg_R2"].iloc[0]:>16.4f} '
      f'{fm2["Avg_R2"].iloc[0]:>16.4f} {fm3["Avg_R2"].iloc[0]:>16.4f}')
print('=' * 76)
print('Coefficients \u00d7 100 (monthly return in percent).')
print('t-statistics in parentheses. *** p<0.01, ** p<0.05, * p<0.10.')

## 9. Calendar-Time Long/Short Portfolio

Each month, firms are ranked on their most recently filed semantic drift signal.  
Equal-weighted Q5 (high drift) and Q1 (low drift) portfolios are formed.  
Monthly portfolio returns are regressed on the Carhart (1997) four-factor model:

$$R^{L/S}_t - R^f_t = \alpha + \beta_1 \text{MKT}_t + \beta_2 \text{SMB}_t + \beta_3 \text{HML}_t + \beta_4 \text{UMD}_t + \varepsilon_t$$

HAC (Newey-West) standard errors with 6 lags.

In [ ]:
# Monthly equal-weighted portfolio returns per quintile
monthly_port = (
    panel.groupby(['return_month', 'quintile'])['monthly_return']
         .mean().unstack('quintile').sort_index()
)
monthly_port.columns = [f'Q{int(c)}' for c in monthly_port.columns]
monthly_port['LS'] = monthly_port.get('Q5', np.nan) - monthly_port.get('Q1', np.nan)
monthly_port.index = pd.to_datetime(monthly_port.index)

# ── Load Fama-French + Momentum factors from Kenneth French data library ──
import pandas_datareader.data as web

try:
    ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench',
                         start='2005-01', end='2025-12')[0] / 100
    umd = web.DataReader('F-F_Momentum_Factor', 'famafrench',
                         start='2005-01', end='2025-12')[0] / 100
    ff3.index = pd.PeriodIndex(ff3.index, freq='M').to_timestamp()
    umd.index = pd.PeriodIndex(umd.index, freq='M').to_timestamp()
    factors = ff3.join(umd, how='inner')
    factors.columns = ['MKT', 'SMB', 'HML', 'RF', 'UMD']
    HAS_FACTORS = True
    print(f'FF4 factors: {len(factors)} months '
          f'({factors.index.min().date()} \u2013 {factors.index.max().date()})')
except Exception as exc:
    print(f'WARNING: could not load FF factors ({exc}). Alpha table skipped.')
    HAS_FACTORS = False

In [ ]:
if HAS_FACTORS:
    joined = monthly_port.join(factors, how='inner').dropna(subset=['RF'])

    def four_factor_reg(excess_ret: pd.Series, nw_lags: int = 6) -> dict:
        y = excess_ret.dropna().values
        X = sm.add_constant(
            joined.loc[excess_ret.dropna().index, ['MKT', 'SMB', 'HML', 'UMD']].values
        )
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})
        return {
            'alpha_m':  res.params[0],
            'alpha_a':  (1 + res.params[0])**12 - 1,
            't_alpha':  res.tvalues[0],
            'p_alpha':  res.pvalues[0],
            'b_mkt':    res.params[1],
            'b_smb':    res.params[2],
            'b_hml':    res.params[3],
            'b_umd':    res.params[4],
            'r2':       res.rsquared,
            'N':        int(res.nobs),
        }

    port_names = ['Q1', 'Q5', 'LS']
    regs = {}
    for name in port_names:
        if name not in joined.columns:
            continue
        exc = joined[name] - joined['RF']
        regs[name] = four_factor_reg(exc)

    print('\nTable 4 \u2014 Carhart Four-Factor Alphas')
    print(f'HAC Newey-West standard errors (6 lags)')
    print('=' * 72)
    print(f'{"":26} {"Q1 (Short)":>14} {"Q5 (Long)":>14} {"L/S Spread":>14}')
    print('-' * 72)

    for label, key, fmt_s in [
        ('Alpha (monthly, %)',  'alpha_m',  '{:.4f}'),
        ('Alpha (annualised, %)','alpha_a', '{:.3f}'),
        ('   [t-statistic]',    't_alpha',  '[{:.2f}]'),
        ('MKT',                 'b_mkt',    '{:.3f}'),
        ('SMB',                 'b_smb',    '{:.3f}'),
        ('HML',                 'b_hml',    '{:.3f}'),
        ('UMD',                 'b_umd',    '{:.3f}'),
        ('R\u00b2',             'r2',       '{:.3f}'),
        ('N (months)',          'N',        '{}'),
    ]:
        vals = []
        for name in port_names:
            if name not in regs:
                vals.append('')
                continue
            v = fmt_s.format(regs[name][key] * (100 if 'alpha' in key and key != 't_alpha' and key != 'p_alpha' else 1))
            if key == 'alpha_m':
                v += stars(regs[name]['p_alpha'])
            vals.append(v)
        print(f'{label:<26} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14}')

    print('=' * 72)
    print('*** p<0.01, ** p<0.05, * p<0.10. Alpha \u00d7 100 (monthly percent).')

## 10. Performance Statistics & Figures

In [ ]:
def perf_stats(ret: pd.Series, rf: pd.Series = None) -> dict:
    r = ret.dropna()
    rf_ = (rf.reindex(r.index).fillna(0) if rf is not None
           else pd.Series(0.0, index=r.index))
    exc = r - rf_
    ann_ret = (1 + r.mean())**12 - 1
    ann_vol = r.std() * np.sqrt(12)
    sharpe  = exc.mean() / r.std() * np.sqrt(12)
    cum     = (1 + r).cumprod()
    dd      = (cum / cum.cummax() - 1).min()
    calmar  = ann_ret / abs(dd) if dd != 0 else np.nan
    return {
        'Ann. Return (%)': round(ann_ret * 100, 2),
        'Ann. Vol (%)':    round(ann_vol * 100, 2),
        'Sharpe Ratio':    round(sharpe,  3),
        'Max Drawdown (%)':round(dd * 100, 2),
        'Calmar Ratio':    round(calmar,   3) if not np.isnan(calmar) else np.nan,
        'Skewness':        round(r.skew(), 3),
        'Excess Kurtosis': round(r.kurtosis(), 3),
        'N (months)':      len(r),
    }

rf_s = factors['RF'] if HAS_FACTORS else None

perf_rows = {}
for col in ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'LS']:
    if col in monthly_port.columns:
        perf_rows[col] = perf_stats(monthly_port[col], rf_s)

perf_df = pd.DataFrame(perf_rows).T
print('\nTable 5 \u2014 Portfolio Performance Statistics')
print(perf_df.to_string())
print('Sharpe ratio annualised (\u00d7\u221212). Max drawdown from peak.')

In [ ]:
# ── Figure 3: Cumulative returns ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for col, color, lw in [('Q1', '#d62728', 1.2), ('Q3', '#7f7f7f', 1.0),
                        ('Q5', '#1f77b4', 1.2), ('LS', '#2ca02c', 2.0)]:
    if col not in monthly_port.columns:
        continue
    cum = (1 + monthly_port[col].dropna()).cumprod()
    ls  = 'solid' if col == 'LS' else 'dashed'
    lbl = 'L/S (Q5\u2212Q1)' if col == 'LS' else col
    ax.plot(cum.index, cum.values, label=lbl, color=color, linewidth=lw, linestyle=ls)
ax.axhline(1, color='black', linewidth=0.6, linestyle=':')
ax.set_title('Figure 3 \u2014 Cumulative Portfolio Returns', fontsize=11)
ax.set_ylabel('Cumulative return (\u00d71)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(fontsize=9); ax.grid(linestyle='--', alpha=0.3)

# ── Figure 4: Rolling 36-month Sharpe ratio ──────────────────────────
ax = axes[1]
if 'LS' in monthly_port.columns:
    ls_ret = monthly_port['LS'].dropna()
    roll_sharpe = (
        ls_ret.rolling(36).mean() /
        ls_ret.rolling(36).std() * np.sqrt(12)
    )
    ax.plot(roll_sharpe.index, roll_sharpe.values, color='#2ca02c', linewidth=1.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.fill_between(roll_sharpe.index, roll_sharpe.values, 0,
                    where=roll_sharpe.values >= 0, alpha=0.2, color='#2ca02c')
    ax.fill_between(roll_sharpe.index, roll_sharpe.values, 0,
                    where=roll_sharpe.values < 0, alpha=0.2, color='#d62728')
    ax.set_title('Figure 4 \u2014 Rolling 36-Month Sharpe Ratio (L/S)', fontsize=11)
    ax.set_ylabel('Annualised Sharpe Ratio')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'fig34_portfolio_performance.pdf'),
            bbox_inches='tight')
plt.show()

## 11. LaTeX Table Export

In [ ]:
def to_latex_fm(fm_results: list, labels: list, var_order: list,
                var_labels: dict, caption: str, label: str) -> str:
    ncols = len(fm_results)
    col_spec = 'l' + 'r' * ncols
    col_header = ' & '.join([f'({i+1})' for i in range(ncols)])

    lines = [
        r'\begin{table}[htbp]',
        r'\centering',
        f'\\caption{{{caption}}}',
        f'\\label{{{label}}}',
        f'\\begin{{tabular}}{{{col_spec}}}',
        r'\toprule',
        f'Variable & {col_header} \\\\',
        r'\midrule',
    ]

    for var in var_order:
        vl = var_labels.get(var, var).replace('&', '\\&')
        coef_parts, tstat_parts = [], []
        for fm in fm_results:
            row = fm[fm['Variable'] == var]
            if len(row) == 0:
                coef_parts.append('')
                tstat_parts.append('')
            else:
                c = row.iloc[0]
                st = stars(c['p_val'])
                coef_parts.append(f'{c["Coef"]*100:.4f}{st}')
                tstat_parts.append(f'({c["t_stat"]:.2f})')
        lines.append(f'{vl} & {" & ".join(coef_parts)} \\\\')
        lines.append(f' & {" & ".join(tstat_parts)} \\\\')

    lines += [
        r'\midrule',
        f'N months & ' + ' & '.join([str(int(fm["N_months"].iloc[0])) for fm in fm_results]) + r' \\',
        f'Avg N & '    + ' & '.join([str(int(fm["Avg_N"].iloc[0]))    for fm in fm_results]) + r' \\',
        f'Avg $R^2$ & ' + ' & '.join([f'{fm["Avg_R2"].iloc[0]:.4f}'  for fm in fm_results]) + r' \\',
        r'\bottomrule',
        r'\end{tabular}',
        r'\begin{tablenotes}',
        r'\small',
        r'\item Coefficients $\times$ 100 (monthly return in percent). '
        r't-statistics in parentheses, Newey-West corrected (6 lags). '
        r'*** $p<0.01$, ** $p<0.05$, * $p<0.10$.',
        r'\end{tablenotes}',
        r'\end{table}',
    ]
    return '\n'.join(lines)


latex_t3 = to_latex_fm(
    [fm1, fm2, fm3],
    ['(1) Raw', '(2) Std', '(3) Controls'],
    ['semantic_drift', 'drift_std'] + CONTROLS + ['const'],
    VAR_LABELS,
    caption='Fama-MacBeth Cross-Sectional Regressions',
    label='tab:fm_regressions',
)

latex_path = os.path.join(CONFIG['output_folder'], 'table3_fama_macbeth.tex')
with open(latex_path, 'w') as f:
    f.write(latex_t3)
print(f'Saved: {latex_path}')
print(latex_t3)